# Study 1

Compared satisfaction with a 45-minute point estimate, a narrow range (40–50 minutes), and a moderate range (35–55 minutes) after participants observed delivery outcomes.

In [1]:
# Import packages and locate the data folder.
from pathlib import Path
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import binomtest, t
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/Study 1.xlsx').exists())
os.environ['MPLCONFIGDIR'] = str(ROOT / '.cache/matplotlib')
ITEMS = ['sb_like', 'sb_info', 'sb_useful', 'sb_accurate', 'sb_trust']
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

In [2]:
# Load data, apply exclusions, and calculate satisfaction.
d = pd.read_excel(ROOT / 'data/raw/Study 1.xlsx')
raw_n = len(d)

d = d.loc[(d.ExcludeDuplicateIP != 1) & (d.ExcludeDuplicateID != 1)].copy()
assert len(d) == 698
d['satisfaction'] = d[ITEMS].mean(axis=1)
rows, descriptions = [], []
display(pd.DataFrame([{'raw_n':raw_n, 'replicated_n':len(d), 'paper_n':698, 'match':len(d)==698}]))

,raw_n,replicated_n,paper_n,match
0,700,698,698,True


In [3]:
# Define helpers for regressions and paper comparisons.
def record(study, label, fit, term, target_b=None, target_se=None, sign=1):
    ci = np.sort(np.asarray(fit.conf_int().loc[term]) * sign)
    b, se = (sign * fit.params[term], fit.bse[term])
    row = {'study': study, 'analysis': label, 'n': int(fit.nobs), 'b': b, 'se': se, 'p': fit.pvalues[term], 'ci_low': ci[0], 'ci_high': ci[1], 'published_b': target_b, 'published_se': target_se, 'b_matches_2dp': None if target_b is None else abs(b - target_b) <= 0.0050001, 'se_matches_2dp': None if target_se is None else abs(se - target_se) <= 0.0050001}
    rows.append(row)

def ols(d, y, terms):
    return sm.OLS(d[y], sm.add_constant(d[terms].astype(float)), missing='drop').fit()

def contrast(d, study, labels, treatment, reference, target_b, target_se, label=None, covariates=()):
    frame = d.copy()
    for name, code in labels.items():
        frame[name] = (frame.Condition == code).astype(float)
    terms = [name for name in labels if name != reference] + list(covariates)
    record(study, label or f'{treatment} - {reference}', ols(frame, 'satisfaction', terms), treatment, target_b, target_se)

In [4]:
# Estimate satisfaction differences between the three estimate formats.
labels = {'point': 1, 'narrow': 2, 'moderate': 3}
for a,b,beta,se in [('moderate', 'point', 0.56, 0.1), ('narrow', 'point', 0.5, 0.1), ('moderate', 'narrow', 0.06, 0.1)]:
    contrast(d,1,labels,a,b,beta,se)
published = {1: (4.81, 1.28), 2: (5.32, 1.0), 3: (5.37, 1.08)}

In [5]:
# Display estimates beside the paper and check agreement at reported precision.
results = pd.DataFrame(rows)
checked = results.published_b.notna()
assert results.loc[checked, 'b_matches_2dp'].all()
assert results.loc[checked, 'se_matches_2dp'].all()
display(results[['analysis','n','b','published_b','se','published_se','p','b_matches_2dp','se_matches_2dp']])

,analysis,n,b,published_b,se,published_se,p,b_matches_2dp,se_matches_2dp
0,moderate - point,698,0.5622,0.56,0.1045,0.1,1.0226e-07,True,True
1,narrow - point,698,0.5030,0.50,0.1045,0.1,1.8286e-06,True,True
2,moderate - narrow,698,0.0592,0.06,0.1044,0.1,5.7067e-01,True,True


In [6]:
# Compare group means and standard deviations with the published table.
table = d.groupby('Condition').satisfaction.agg(n='count',mean='mean',sd='std')
table['condition'] = table.index.map({v:k for k,v in labels.items()})
table['paper_mean'] = [published[c][0] for c in table.index]
table['paper_sd'] = [published[c][1] for c in table.index]
table['match'] = ((table['mean']-table.paper_mean).abs()<=.0050001)&((table.sd-table.paper_sd).abs()<=.0050001)
assert table['match'].all()
display(table)

,n,mean,sd,condition,paper_mean,paper_sd,match
Condition,,,,,,,
1,232,4.8121,1.2844,point,4.81,1.28,True
2,233,5.3150,1.0008,narrow,5.32,1.00,True
3,233,5.3742,1.0767,moderate,5.37,1.08,True
